# TaskGen-ID — Google Colab

Jalankan generator pertanyaan **bahasa Indonesia saja** dari HP. Notebook ini memakai Gemini API dan menyimpan hasil ke Google Drive.

## Sebelum mulai
1. Buka notebook ini di Google Colab.
2. Di panel **Secrets** (ikon kunci), tambahkan secret bernama `GEMINI_API_KEY` dengan API key dari Google AI Studio.
3. Aktifkan akses notebook untuk secret tersebut.
4. Jalankan sel dari atas ke bawah.

API key tidak disimpan di notebook dan tidak dicetak ke output. Jangan pernah menulis API key langsung di sel kode atau commit key ke GitHub.

**Catatan:** penggunaan Gemini API tunduk pada kuota dan ketentuan akun Google AI Studio. Colab gratis tidak membuat API Gemini otomatis tanpa batas.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from google.colab import drive, userdata

print("Runtime siap.")

## 1. Hubungkan Google Drive

File hasil akan disimpan di `MyDrive/taskgen-id/outputs/`.

In [ ]:
drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/taskgen-id/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Folder output siap: {OUTPUT_DIR}")

## 2. Ambil API key dari Colab Secrets

Pastikan secret bernama `GEMINI_API_KEY` sudah dibuat dan akses notebook diaktifkan.

In [ ]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError(
        "Secret GEMINI_API_KEY tidak ditemukan. Tambahkan di panel Colab Secrets "
        "dan aktifkan akses untuk notebook ini."
    )

print("Secret Gemini ditemukan. Nilainya tidak ditampilkan.")

## 3. Siapkan Rust dan unduh kode

Notebook ini menjalankan proyek Rust aslinya; bukan salinan Python terpisah.

In [ ]:
RUST_INSTALL = "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"
subprocess.run(["bash", "-lc", RUST_INSTALL], check=True)

CARGO = Path.home() / ".cargo" / "bin" / "cargo"
os.environ["PATH"] = f"{Path.home() / '.cargo' / 'bin'}:{os.environ.get('PATH', '')}"
subprocess.run([str(CARGO), "version"], check=True)

REPO_DIR = Path("/content/taskgen-id")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/IND-NV/taskgen-id.git", str(REPO_DIR)],
    check=True,
)
print("Kode berhasil diunduh.")

## 4. Konfigurasi generator

Nilai awal mengikuti konfigurasi default proyek. Kalau ingin uji pertama lebih ringan, ubah `COUNT` menjadi 5 atau 10.

In [ ]:
COUNT = 250
WORKERS = 5
TEMPERATURE = 0.9
MODEL = "gemini-3.8-flash"
OUTPUT_NAME = "questions.jsonl"

# False = mulai file baru; True = lanjutkan dengan menambahkan ke file yang sudah ada.
APPEND_EXISTING = False

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
print(f"Model: {MODEL}")
print(f"Jumlah pertanyaan: {COUNT}")
print(f"File output: {OUTPUT_PATH}")

## 5. Jalankan generator

Hasil utama berupa JSONL. Program juga membuat laporan bernama `questions_README.md` di folder yang sama. Data ditulis bertahap agar hasil yang sudah selesai tetap tersimpan jika sesi terputus.

In [ ]:
env = os.environ.copy()
env["GEMINI_API_KEY"] = GEMINI_API_KEY

cmd = [
    str(CARGO), "run", "--release", "--",
    "--api-base", "https://generativelanguage.googleapis.com/v1beta/openai",
    "--model", MODEL,
    "--count", str(COUNT),
    "--workers", str(WORKERS),
    "--temperature", str(TEMPERATURE),
    "--output", str(OUTPUT_PATH),
]
if APPEND_EXISTING:
    cmd.append("--append")

print("Mulai menjalankan TaskGen-ID. API key tidak dicetak.")
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

print(f"Selesai. Periksa hasil di: {OUTPUT_DIR}")

## Lokasi hasil

Buka Google Drive → **My Drive → taskgen-id → outputs**. File `questions.jsonl` berisi pertanyaan; `questions_README.md` berisi laporan run.

Kalau sesi Colab terputus, buka kembali notebook dan jalankan lagi. Untuk melanjutkan ke file yang sama, set `APPEND_EXISTING = True` sebelum menjalankan sel generator. Pastikan nama file sama agar hasil ditambahkan ke file yang benar.